In [ ]:
import pandas as pd
import json
import os
import csv

# --- File paths ---
input_file = ""
output_dir = ""

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

#  Chunk size for reading CSV 
chunk_size = 300000

#  Function to build product info 
def build_product_info(row):
    return {
        "product_id": row["product_id"],
        "category_id": row["category_id"],
        "category_code": row["category_code"] if pd.notna(row["category_code"]) else None,
        "brand": row["brand"] if pd.notna(row["brand"]) else None,
        "price": row["price"]
    }

#  Read CSV in chunks
chunk_iter = pd.read_csv(
    input_file,
    chunksize=chunk_size,
    quoting=csv.QUOTE_MINIMAL,
    on_bad_lines="skip",
    engine="python"
)

total_rows = 0

# Process each chunk
for i, chunk in enumerate(chunk_iter):
    print(f"Processing chunk {i}... Rows: {len(chunk)}")
    total_rows += len(chunk)

    # Build product_info column
    chunk["product_info"] = chunk.apply(build_product_info, axis=1)

    # Select the columns we want
    df_final = chunk[[
        "event_time",
        "event_type",
        "user_id",
        "user_session",
        "product_info"
    ]]

    # Convert DataFrame to dict records
    records = df_final.to_dict(orient="records")

    # Write one JSON object per line
    output_file = os.path.join(output_dir, f"Feb_data_part_{i}.json")
    with open(output_file, "w", encoding="utf-8") as f:
        for record in records:
            json.dump(record, f)
            f.write("\n")  # newline after each record

print("Total processed rows:", total_rows)
print("Done!")